# PolyAI Production Training (Tesla T4)

**Expected Performance:**
- CPU: ~17ms/inference → 71s per 80 moves
- T4 GPU: ~1-2ms/inference → **5-10s per 80 moves** (10-15x faster!)

In [ ]:
# Training config
ITERATIONS = 100      # Total training iterations
GAMES_PER_ITER = 30   # Games generated each iteration
MCTS_ITERATIONS = 100 # MCTS depth (higher = better quality, slower)

os.environ['MCTS_ITERS'] = str(MCTS_ITERATIONS)

# Create directories
!mkdir -p archive checkpoints

print(f"Will run {ITERATIONS} iterations")
print(f"Each iteration: {GAMES_PER_ITER} games × {MCTS_ITERATIONS} MCTS iters")
print(f"Estimated time per iteration: ~{GAMES_PER_ITER * 10 / 60:.1f} min (on T4)")

### Run Training Loop

In [ ]:
import subprocess
import time
import re
from IPython.display import clear_output

# Training log
metrics_log = []

for iteration in range(1, ITERATIONS + 1):
    print(f"\n{'='*50}")
    print(f"Iteration {iteration}/{ITERATIONS}")
    print(f"{'='*50}\n")
    
    # 1. Self-Play
    print("[Self-Play] Generating games...")
    start_time = time.time()
    
    env = os.environ.copy()
    env['NUM_GAMES'] = str(GAMES_PER_ITER)
    
    sp_result = subprocess.run(
        ['./target/release/self_play'],
        capture_output=True,
        text=True,
        env=env
    )
    
    sp_time = time.time() - start_time
    print(sp_result.stdout)
    
    # Extract metrics
    metrics_match = re.search(r'"avg_score": ([0-9.]+), "max_score": ([0-9]+)', sp_result.stdout)
    if metrics_match:
        avg_score = float(metrics_match.group(1))
        max_score = int(metrics_match.group(2))
    else:
        avg_score, max_score = 0, 0
    
    print(f"Self-play took {sp_time:.1f}s")
    
    # 2. Training
    print("\n[Training] Updating model...")
    train_result = subprocess.run(
        ['python3', 'train.py'],
        capture_output=True,
        text=True
    )
    
    print(train_result.stdout)
    
    # Extract loss
    loss_match = re.search(r'"loss": ([0-9.]+)', train_result.stdout)
    loss = float(loss_match.group(1)) if loss_match else 0.0
    
    # 3. Log
    metrics_log.append({
        'iteration': iteration,
        'avg_score': avg_score,
        'max_score': max_score,
        'loss': loss,
        'time': sp_time
    })
    
    print(f"\n✅ Iteration {iteration} complete")
    print(f"   Avg Score: {avg_score:.2f} | Max: {max_score} | Loss: {loss:.4f}")
    
    # 4. Cleanup
    !mv games_*.safetensors archive/ 2>/dev/null || true
    
    # 5. Checkpoint every 10 iterations
    if iteration % 10 == 0:
        !cp model.safetensors checkpoints/model_iter_{iteration}.safetensors
        print(f"   💾 Checkpoint saved")
    
    # Show progress
    if len(metrics_log) > 1:
        recent = metrics_log[-5:]
        avg_recent_score = sum(m['avg_score'] for m in recent) / len(recent)
        print(f"   📈 Recent avg score trend: {avg_recent_score:.2f}")

print("\n" + "="*50)
print("✅ Training Complete!")
print("="*50)

## Download Results

In [ ]:
from google.colab import files
import pandas as pd

# Download trained model
files.download('model.safetensors')

# Download metrics as CSV
df = pd.DataFrame(metrics_log)
df.to_csv('training_metrics.csv', index=False)
files.download('training_metrics.csv')

# Download all checkpoints
!zip -r checkpoints.zip checkpoints/
files.download('checkpoints.zip')

## Plot Training Progress

In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame(metrics_log)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df['iteration'], df['avg_score'])
axes[0].set_title('Average Score')
axes[0].set_xlabel('Iteration')

axes[1].plot(df['iteration'], df['max_score'])
axes[1].set_title('Max Score')
axes[1].set_xlabel('Iteration')

axes[2].plot(df['iteration'], df['loss'])
axes[2].set_title('Training Loss')
axes[2].set_xlabel('Iteration')

plt.tight_layout()
plt.show()